# Checkpoint run — Sai Chetan Muppalla: live Full CoT and DEER

Runs DeepSeek-R1-Distill-Qwen-1.5B on the first 100 GSM8K and first 100 MATH-500 questions:
1. **Full CoT**: no early exit (baseline)
2. **DEER τ = 0.95**: official settings for R1-Distill (`--think_ratio 0.6 --policy avg1`)
3. *(if time)* **DEER τ = 0.95 with a greedy trial answer** on GSM8K, to measure run-to-run variation

**How to run:** T4 runtime → *Runtime → Run all*. Expected time ~1–1.5 h (+20 min for step 8).
Keep the tab open. If Colab disconnects, reconnect and *Run all* again.

## 1. GPU check
*Runtime → Change runtime type → T4 GPU* first. This should print `Tesla T4`.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Google Drive backup
Outputs are copied to Drive after every run, so a Colab disconnect loses nothing.
If you reconnect, just run all cells again: finished runs are restored from Drive and skipped.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 3. Get the code

In [3]:
REPO = "https://github.com/PeramVishnuSree/Adaptive-Early-Exit-for-Efficient-Reasoning-Reproducing-and-Extending-DEER.git"
!git clone -q $REPO /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo/Vishnu-evaluation
!git log --oneline -1

/content/repo/Vishnu-evaluation
e7758d6 (HEAD -> main, origin/main, origin/HEAD) Add checkpoint 3 notebook: never-exit trajectory runs


## 4. Install
vLLM 0.26.0 matches Colab's Python 3.13 / torch 2.11. Exit logic and decoding settings are the official DEER ones.

In [4]:
!pip install -q "vllm==0.26.0" nltk
!pip install -q "antlr4-python3-runtime==4.11.0" "latex2sympy2_extended[antlr4_11_0]" regex word2number

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Check the install. Expect `vllm 0.26.0 | torch 2.11.0+... | cuda True`, then a few generated words.
The first model load takes a few minutes.

In [5]:
!python -c "import vllm, torch, sympy; print('vllm', vllm.__version__, '| torch', torch.__version__, '| sympy', sympy.__version__, '| cuda', torch.cuda.is_available())"
!python -c "from vllm import LLM, SamplingParams; llm = LLM('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', dtype='float16', max_model_len=4096, gpu_memory_utilization=0.8); print(repr(llm.generate(['What is 2+3?'], SamplingParams(max_tokens=20, temperature=0))[0].outputs[0].text))"

vllm 0.26.0 | torch 2.11.0+cu130 | sympy 1.14.0 | cuda True
INFO 11-03 05:42:59 [api_utils.py:273] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
config.json: 100% 679/679 [00:00<00:00, 2.01MB/s]
INFO 11-03 05:43:18 [model.py:623] Resolved architecture: Qwen2ForCausalLM
WARNING 11-03 05:43:18 [model.py:2123] Casting torch.bfloat16 to torch.float16.
INFO 11-03 05:43:18 [model.py:1788] Using max model len 4096
INFO 11-03 05:43:18 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-03 05:43:18 [vllm.py:1109] Asynchronous scheduling is enabled.
INFO 11-03 05:43:18 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
tokenizer_config.json: 100% 3.07k/3.07k [00:00<00:00, 5.55MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 133MB/s]
[transfor

**Only if the check fails with a CUDA error** (`undefined symbol`, `libcudart.so.12`, ...): set `FIX_CUDA = True`,
run this cell, then *Runtime → Restart session*, and run the check cell again.

In [6]:
FIX_CUDA = False
if FIX_CUDA:
    !pip install -q "torch==2.11.0" "torchvision==0.26.0" "torchaudio==2.11.0" --index-url https://download.pytorch.org/whl/cu129

## 5. Settings


In [7]:
import os, glob, json, shutil
%cd /content/repo/Vishnu-evaluation
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
N = 100                 # first 100 questions of each test set
MAX_TOKENS = 8000       # total generation budget (paper uses 16000; reduced to fit the time)
DATASETS = ["gsm8k", "math"]
DTYPE = "float16"       # T4 has no bf16

OUT = "outputs/ckpt"
DRIVE_DIR = "/content/drive/MyDrive/deer_ckpt/person2"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copytree(DRIVE_DIR, OUT, dirs_exist_ok=True)   # restore finished runs after a disconnect

def done(path):
    """A run is finished once its .meta.json exists (written last)."""
    return os.path.exists(path[: -len(".jsonl")] + ".meta.json")

def backup():
    shutil.copytree(OUT, DRIVE_DIR, dirs_exist_ok=True)
    print("backed up to", DRIVE_DIR)

print("already finished:", [os.path.basename(f) for f in sorted(glob.glob(f"{OUT}/*.meta.json"))])

/content/repo/Vishnu-evaluation
already finished: []


## 6. Full CoT baseline

In [8]:
for ds in DATASETS:
    out = f"{OUT}/fullcot_{ds}.jsonl"
    if done(out):
        print("skip (done):", out); continue
    !python scripts/run_full_cot.py --model $MODEL --dataset $ds --limit $N \
        --max_generated_tokens $MAX_TOKENS --dtype $DTYPE --output_file $out
    backup()

[transformers] `tokenizer_class` is `LlamaTokenizerFast` but `tokenizer.json` is ByteLevel; loading `TokenizersBackend`. Set `tokenizer_class` to `TokenizersBackend` in `tokenizer_config.json`.
INFO 11-03 05:46:05 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 12000, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
INFO 11-03 05:46:07 [model.py:623] Resolved architecture: Qwen2ForCausalLM
WARNING 11-03 05:46:07 [model.py:2123] Casting torch.bfloat16 to torch.float16.
INFO 11-03 05:46:07 [model.py:1788] Using max model len 12000
INFO 11-03 05:46:07 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-03 05:46:07 [vllm.py:1109] Asynchronous scheduling is enabled.
INFO 11-03 05:46:07 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=

## 7. DEER, τ = 0.95

In [9]:
for ds in DATASETS:
    out = f"{OUT}/deer_t0.95_{ds}.jsonl"
    if done(out):
        print("skip (done):", out); continue
    !python third_party/deer/vllm_deer_logged.py --model_name_or_path $MODEL --dataset_dir data --dataset $ds --limit $N --max_generated_tokens $MAX_TOKENS --think_ratio 0.6 --policy avg1 --dtype $DTYPE --gpu-memory-utilization 0.9 \
        --threshold 0.95 --output_file $out 2>&1 | grep -v "Early exit threshold not reached" | tail -5
    backup()

Question 93: Actually reached thinking limit (4800/4800). Switching to answer phase.

All questions processed, saving results...
Evaluation completed! Attempted to process 100 questions in total, successfully recorded 100 results, took 389.10 seconds
Results saved to: outputs/ckpt/deer_t0.95_gsm8k.jsonl
backed up to /content/drive/MyDrive/deer_ckpt/person2
Question 97: Actually reached thinking limit (4800/4800). Switching to answer phase.

All questions processed, saving results...
Evaluation completed! Attempted to process 100 questions in total, successfully recorded 100 results, took 649.93 seconds
Results saved to: outputs/ckpt/deer_t0.95_math.jsonl
backed up to /content/drive/MyDrive/deer_ckpt/person2


## 8. (If time) Variance check: DEER with a greedy trial answer, GSM8K only
Official DEER samples the trial answer at temperature 1.0, so its exits are partly random. This reruns GSM8K
with temperature 0 for the trial answer. Set `RUN_VARIANCE = False` to skip.

In [10]:
RUN_VARIANCE = True
if RUN_VARIANCE:
    ds = "gsm8k"
    out = f"{OUT}/deer_t0.95_greedyprobe_{ds}.jsonl"
    if done(out):
        print("skip (done):", out)
    else:
        !python third_party/deer/vllm_deer_logged.py --model_name_or_path $MODEL --dataset_dir data --dataset $ds --limit $N --max_generated_tokens $MAX_TOKENS --think_ratio 0.6 --policy avg1 --dtype $DTYPE --gpu-memory-utilization 0.9 \
            --threshold 0.95 --trial_temperature 0 --output_file $out 2>&1 | grep -v "Early exit threshold not reached" | tail -5
        backup()

backed up to /content/drive/MyDrive/deer_ckpt/person2


## Score
Grades every run in `outputs/ckpt` with DEER's grader and prints a summary.

In [11]:
for f in sorted(glob.glob(f"{OUT}/*.jsonl")):
    if f.endswith(".scored.jsonl"):
        continue
    ds = "gsm8k" if f.endswith("_gsm8k.jsonl") else "math"
    !python scripts/score_run.py --input $f --dataset $ds
backup()

rows = []
for f in sorted(glob.glob(f"{OUT}/*.summary.json")):
    s = json.load(open(f))
    rows.append((s["input"], s["n"], 100 * s["accuracy"], s["avg_output_tokens"], s["avg_compute_tokens"], (s.get("wall_time_s") or 0) / 60))
print(f"{'run':40s} {'n':>4s} {'acc%':>6s} {'out tok':>8s} {'compute':>8s} {'min':>6s}")
for r in rows:
    print(f"{r[0]:40s} {r[1]:4d} {r[2]:6.1f} {r[3]:8.0f} {r[4]:8.0f} {r[5]:6.1f}")

deer_t0.95_gsm8k.jsonl  n=100
  accuracy           68.0% (+/- 9.1)
  avg reasoning tok  442
  avg output tok     773
  avg compute tok    778
  exit reasons       {'natural': 66, 'budget': 3, 'confident': 31}
  wrote outputs/ckpt/deer_t0.95_gsm8k.scored.jsonl and .summary.json
deer_t0.95_math.jsonl  n=100
  accuracy           69.0% (+/- 9.1)
  avg reasoning tok  1423
  avg output tok     1747
  avg compute tok    1763
  exit reasons       {'confident': 78, 'budget': 13, 'natural': 9}
  wrote outputs/ckpt/deer_t0.95_math.scored.jsonl and .summary.json
fullcot_gsm8k.jsonl  n=100
  accuracy           77.0% (+/- 8.2)
  avg reasoning tok  1229
  avg output tok     1452
  avg compute tok    1452
  exit reasons       {'natural': 89, 'budget': 11}
  wrote outputs/ckpt/fullcot_gsm8k.scored.jsonl and .summary.json
fullcot_math.jsonl  n=100
  accuracy           66.0% (+/- 9.3)
  avg reasoning tok  3665
  avg output tok     3894
  avg compute tok    3894
  exit reasons       {'natural': 68, 'budge

## Save and send
Downloads `ckpt_SaiC_live.zip`.

In [17]:
!zip -qr ckpt_person2_live.zip outputs/ckpt
from google.colab import files
files.download("ckpt_SaiC_live.zip")


zip error: Nothing to do! (try: zip -qr ckpt_person2_live.zip . -i outputs/ckpt)


FileNotFoundError: Cannot find file: ckpt_SaiC_live.zip

In [13]:
# 1) Refresh repo so never-exit scored JSONL is present
%cd /content/repo
!git pull -q
!ls Vishnu-evaluation/results/checkpoint3/never_exit/
# expect: never_gsm8k.scored.jsonl  never_math.scored.jsonl  ...

/content/repo
never_gsm8k.jsonl	  never_gsm8k.summary.json  never_math.scored.jsonl
never_gsm8k.meta.json	  never_math.jsonl	    never_math.summary.json
never_gsm8k.scored.jsonl  never_math.meta.json


In [14]:

from google.colab import files
uploaded = files.upload()  # choose run_controller_real_data.py
!mv run_controller_real_data.py Saichetan-adaptive/scripts/

Saving run_controller_real_data.py to run_controller_real_data.py


In [16]:
# 4) Run from Saichetan-adaptive (default data path points at Vishnu's never_exit)
%cd /content/repo/Saichetan-adaptive
!python scripts/run_controller_real_data.py

/content/repo/Saichetan-adaptive
1406 checkpoints from 133 questions; 200 questions in total
# AdaptiveDEER-MLP on real trajectories (held-out questions)

Mean ± std over 5 random 70/30 question splits. Accuracy uses DEER's grader; tokens are average output tokens per question. Early exits are scored on the trial answer.

Checkpoint-level ROC-AUC (predicting whether the trial answer is correct): MLP 0.920 ± 0.025, DEER confidence 0.929 ± 0.028

## GSM8K (~28 held-out questions per split)

| Policy | Accuracy (%) | Output tokens | Token reduction vs. full |
|---|---|---|---|
| Full reasoning | 79.3 ± 13.2 | 1092 ± 196 | 0% |
| DEER tau=0.8 | 83.9 ± 10.5 | 489 ± 23 | 55% |
| DEER tau=0.9 | 83.9 ± 10.5 | 498 ± 31 | 54% |
| DEER tau=0.95 | 84.5 ± 10.1 | 522 ± 42 | 52% |
| DEER tau=0.99 | 83.3 ± 11.5 | 671 ± 126 | 39% |
| Consistency (2 in a row) | 83.9 ± 10.5 | 647 ± 84 | 41% |
| Consistency + momentum | 83.9 ± 10.5 | 669 ± 86 | 39% |
| MLP p>=0.3 | 84.4 ± 8.6 | 486 ± 28 | 55% |
| MLP p>=0